# E7 Notebook 26A — CIC-IDS2017 timestamp-format diagnostic

**Why this exists:** The original Notebook 26 stopped on Monday after three Friday CSVs parsed using a single minute-level timestamp format. That strict stop is correct: the dataset might contain heterogeneous string representations. This notebook **does not guess or modify dates**.

**Run this before revising Notebook 26.** It inspects only the first 20 Timestamp strings per CSV, shows five original samples and candidate exact parsing formats, and compares parsed weekdays against source filenames. No raw IP addresses, flow identifiers, attacks or user records are printed. No model training, no GitHub upload, no UNSW official test access.

**Reproducibility:** Notebook 26 remains unmodified; Notebook 26A is an additive, version-controlled diagnosis. After reviewing output, a new Notebook 26B will register per-file formats, full scans, and GitHub auto-sync without changing the earlier benchmark evidence.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT=Path('/kaggle/input/datasets/dsshayan/dscicids')
assert ROOT.is_dir(),'CIC-IDS2017 dataset is not attached to this notebook.'
files=sorted(ROOT.rglob('*.csv'))
assert len(files)==8, f'Expected 8 external CSVs, found {len(files)}'
assert len({str(f.parent) for f in files})==1, 'Found multiple source CSV directories'
assert files[0].parent.name=='TrafficLabelling ', 'Unexpected external source folder'
print('VERIFIED SOURCE DIR:',repr(str(files[0].parent)))

# Diagnostic only. No auto-inference, relabeling, rewriting, network fit, or upload.
# Formats checked against ORIGINAL STRINGS to identify candidates, not lock a format.
formats=[
    '%d/%m/%Y %H:%M',       # day/month/year, 24-hour, minute
    '%d/%m/%Y %H:%M:%S',    # day/month/year, 24-hour, second
    '%d/%m/%Y %I:%M %p',    # day/month/year, 12-hour AM/PM, minute
    '%d/%m/%Y %I:%M:%S %p', # day/month/year, 12-hour AM/PM, second
    '%m/%d/%Y %H:%M',       # month/day/year, 24-hour, minute
    '%m/%d/%Y %H:%M:%S',    # month/day/year, 24-hour, second
    '%m/%d/%Y %I:%M %p',
    '%m/%d/%Y %I:%M:%S %p',
]
print('\nCIC-IDS2017 TIMESTAMP-ONLY DIAGNOSTIC')
print('No IP addresses, flow values, full rows or UNSW test data shown.\n')
for index,p in enumerate(files,1):
    timestamps=pd.read_csv(
        p,nrows=20,dtype='string',encoding='utf-8-sig',
        usecols=lambda header:header.strip().lower()=='timestamp')
    assert timestamps.shape[1]==1, f'{p.name}: expected one Timestamp column'
    original=timestamps.iloc[:,0].astype('string').str.strip()
    print(f'[{index}/8] {p.name}')
    print('  ORIGINAL EXAMPLES:',repr(original.head(5).tolist()))
    expected_day=next((weekday for weekday in ('Monday','Tuesday','Wednesday',
                'Thursday','Friday') if p.name.startswith(weekday)),None)
    valid_candidates=[]
    for fmt in formats:
        parsed=pd.to_datetime(original,format=fmt,errors='coerce',exact=True)
        if len(parsed)>0 and bool(parsed.notna().all()):
            weekdays=sorted(set(parsed.dt.day_name().dropna().tolist()))
            agreement=(expected_day in weekdays and len(weekdays)==1)
            valid_candidates.append((fmt,weekdays,agreement))
    if valid_candidates:
        print('  EXACT SAMPLE PARSERS (format, weekdays, matches filename):')
        for item in valid_candidates:
            print('   ',item)
    else:
        print('  NO CANDIDATE FORMAT parsed all 20 samples. Stop and inspect this file.')
    print()

print('DIAGNOSTIC FINISHED. NO TRAINING AND NO PRIOR RESULTS MODIFIED.')
print('Send the complete output to ChatGPT. Do not rerun the full G0 scan yet.')

## Mandatory stop

The displayed candidate formats are **not yet independently authenticated timestamps** and can be ambiguous for dates such as 7/7/2017. Verify formats with filename-day consistency and broader source inspection before freezing them for full-data parsing.

**Send the complete diagnostic output to ChatGPT.** We will then commit the corrected full-audit Notebook 26B to the same GitHub study folder and continue one notebook at a time.